# KRNEL/MRKOV: Estadistica aplicada desde una plataforma academica

**Demo para la Licenciatura en Estadistica**

Este notebook muestra una clase completa, reproducible y ejecutable desde el navegador:

- trabajo con datos compartidos en JupyterHub;
- analisis exploratorio visual con Python;
- modelos estadisticos interpretables;
- clasificacion probabilistica;
- bootstrap para inferencia;
- companion en R con `tidyverse`, `broom`, `lm()`, `glm()` y `boot`;
- una mini-seccion opcional de Spark **local**, sin Spark distribuido sobre Kubernetes.

## 1. Entorno reproducible

La idea central de MRKOV/KRNEL es que estudiantes y profesores trabajen en el mismo entorno academico sin instalar software localmente. Primero registramos y visualizamos informacion basica del runtime, esto nos demuestra el uso de espacios aislados en un entorno distribuído.

In [ ]:
from pathlib import Path
import getpass
import os
import platform
import socket
import subprocess
import sys

import numpy as np
import pandas as pd

SEED = 20260522
rng = np.random.default_rng(SEED)

print(f"Usuario: {getpass.getuser()}")
print(f"Host/pod: {socket.gethostname()}")
print(f"Python: {sys.version.split()[0]}")
print(f"Plataforma: {platform.platform()}")

try:
    r_version = subprocess.run(["R", "--version"], capture_output=True, text=True, timeout=5)
    print("R:", r_version.stdout.splitlines()[0])
except Exception as exc:
    print("R no disponible desde este kernel:", exc)

## 2. Dataset sintetico de bienestar estudiantil

El archivo esperado es `datasets/encuesta_bienestar_estudiantil.csv` cuando se ejecuta desde `Clases`. Si el CSV no existe, esta celda lo genera de manera deterministica con semilla fija. Esto permite ejecutar la demo incluso si alguien copio solo el notebook.

In [ ]:
DATA_FILE = "encuesta_bienestar_estudiantil.csv"
DATA_CANDIDATES = [
    Path.cwd() / "datasets" / DATA_FILE,
    Path.cwd().parent / "datasets" / DATA_FILE,
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), DATA_CANDIDATES[0])
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

def generar_encuesta_bienestar(path=DATA_PATH, n=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    carreras = np.array(["Estadistica", "Actuaria", "Matematicas", "Economia", "Ciencia de Datos"])
    carrera = rng.choice(carreras, size=n, p=[0.34, 0.18, 0.15, 0.17, 0.16])
    semestre = rng.integers(1, 10, size=n)
    grupos = np.array(["A", "B", "C", "D", "E", "F"])
    grupo = rng.choice(grupos, size=n)
    genero = rng.choice(["mujer", "hombre", "no_binario", "prefiere_no_decir"], size=n, p=[0.48, 0.45, 0.03, 0.04])
    beca = rng.choice(["si", "no"], size=n, p=[0.38, 0.62])
    trabaja = np.where(rng.binomial(1, 0.42, size=n) == 1, rng.normal(18, 7, size=n), 0)
    trabaja = np.clip(trabaja, 0, 38).round(1)
    study_effect = {"Estadistica": 0.8, "Actuaria": 0.4, "Matematicas": 0.7, "Economia": -0.1, "Ciencia de Datos": 0.9}
    carrera_study = np.array([study_effect[c] for c in carrera])
    promedio_previo = np.clip(rng.normal(78, 9, size=n) + 0.75 * semestre + carrera_study * 1.8, 45, 100).round(1)
    uso_redes_horas = np.clip(rng.gamma(2.1, 1.25, size=n) + rng.normal(0, 0.35, n), 0.1, 9).round(1)
    actividad_fisica_min = np.clip(rng.gamma(2.4, 24, size=n) + rng.normal(0, 12, n), 0, 220).round(0).astype(int)
    estres_latente = 4.6 + 0.20 * semestre + 0.055 * trabaja + 0.18 * uso_redes_horas - 0.010 * actividad_fisica_min + rng.normal(0, 1.25, n)
    estres_escala = np.clip(np.rint(estres_latente), 1, 10).astype(int)
    horas_sueno = np.clip(7.35 - 0.12 * (estres_escala - 5) - 0.025 * trabaja + rng.normal(0, 0.9, n), 3.6, 10.6).round(1)
    cafe_tazas = np.clip(rng.poisson(np.clip(1.0 + 0.18 * estres_escala + 0.020 * trabaja - 0.10 * horas_sueno, 0.2, 4.8)), 0, 8)
    horas_estudio = 5.0 + 0.42 * semestre + carrera_study + 0.065 * (promedio_previo - 75) - 0.105 * trabaja - 0.20 * uso_redes_horas + rng.normal(0, 2.2, n)
    horas_estudio = np.clip(horas_estudio, 0.5, 22).round(1)
    asistencia_pct = 72 + 1.55 * horas_estudio - 1.45 * estres_escala - 0.16 * trabaja + 0.12 * actividad_fisica_min + rng.normal(0, 8.5, n)
    asistencia_pct = np.clip(asistencia_pct, 35, 100).round(1)
    grade_effect = {"Estadistica": 1.5, "Actuaria": 0.5, "Matematicas": -0.6, "Economia": -1.0, "Ciencia de Datos": 1.1}
    group_effect = dict(zip(grupos, rng.normal(0, 1.8, len(grupos))))
    sleep_penalty = 1.95 * (horas_sueno - 7.25) ** 2
    stress_extreme_penalty = np.where(estres_escala >= 8, 2.8 * (estres_escala - 7), 0)
    calificacion_final = (8.5 + 1.75 * horas_estudio + 0.24 * asistencia_pct + 0.50 * promedio_previo - 1.12 * estres_escala - sleep_penalty - stress_extreme_penalty - 0.13 * trabaja + 0.32 * semestre + np.array([grade_effect[c] for c in carrera]) + np.array([group_effect[g] for g in grupo]) + rng.normal(0, 6.2, n))
    calificacion_final = np.clip(calificacion_final, 35, 100).round(1)
    df_new = pd.DataFrame({
        "student_id": [f"STU-{i:05d}" for i in range(1, n + 1)],
        "carrera": carrera,
        "semestre": semestre,
        "horas_sueno": horas_sueno,
        "horas_estudio": horas_estudio,
        "asistencia_pct": asistencia_pct,
        "actividad_fisica_min": actividad_fisica_min,
        "cafe_tazas": cafe_tazas,
        "estres_escala": estres_escala,
        "uso_redes_horas": uso_redes_horas,
        "promedio_previo": promedio_previo,
        "calificacion_final": calificacion_final,
        "aprobacion": (calificacion_final >= 70).astype(int),
        "beca": beca,
        "trabaja": trabaja,
        "genero": genero,
        "grupo": grupo,
    })
    df_new.to_csv(path, index=False)
    return df_new

if not DATA_PATH.exists():
    df_generado = generar_encuesta_bienestar(DATA_PATH)
    print(f"Dataset generado: {DATA_PATH} ({len(df_generado):,} filas)")
else:
    print(f"Dataset encontrado: {DATA_PATH}")

## 3. Carga, diccionario y validaciones

Antes de modelar, verificamos dimensiones, tipos de datos y valores faltantes. Esta es una practica sencilla pero poderosa: muchas conclusiones fragiles empiezan con datos no revisados.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

In [ ]:
diccionario = pd.DataFrame({
    "variable": ["student_id", "carrera", "semestre", "horas_sueno", "horas_estudio", "asistencia_pct", "actividad_fisica_min", "cafe_tazas", "estres_escala", "uso_redes_horas", "promedio_previo", "calificacion_final", "aprobacion", "beca", "trabaja", "genero", "grupo"],
    "descripcion": ["Identificador anonimo sintetico", "Programa academico", "Semestre cursado", "Horas de sueno por noche", "Horas de estudio semanal", "Porcentaje de asistencia", "Minutos semanales de actividad fisica", "Tazas de cafe al dia", "Escala de estres de 1 a 10", "Horas diarias en redes sociales", "Promedio previo", "Calificacion final", "1 si calificacion_final >= 70", "Cuenta con beca", "Horas de trabajo remunerado por semana", "Genero reportado", "Grupo academico"]
})
diccionario

In [ ]:
validacion = pd.DataFrame({"tipo": df.dtypes.astype(str), "nulos": df.isna().sum(), "n_unicos": df.nunique()})
validacion

## 4. Exploracion visual en Python

Usaremos `seaborn` y `matplotlib` con una estetica consistente. El objetivo no es decorar: es hacer visibles patrones, dispersion, sesgos y relaciones no lineales.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
PALETTE = ["#31572c", "#4f772d", "#90a955", "#132a13", "#ecf39e"]
sns.set_palette(PALETTE)
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 12

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.histplot(df["calificacion_final"], bins=34, kde=True, color="#31572c", ax=ax)
ax.axvline(70, color="#bc4749", linestyle="--", linewidth=2, label="Umbral de aprobacion")
ax.set_title("Distribucion de la calificacion final")
ax.set_xlabel("Calificacion final")
ax.set_ylabel("Numero de estudiantes")
ax.legend()
sns.despine()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sample = df.sample(1200, random_state=SEED)
sns.regplot(data=sample, x="horas_estudio", y="calificacion_final", scatter_kws={"alpha": 0.28, "s": 28, "color": "#31572c"}, line_kws={"color": "#bc4749", "linewidth": 3}, lowess=True, ax=ax)
ax.set_title("Mas estudio suele asociarse con mayor desempeno, con variabilidad realista")
ax.set_xlabel("Horas de estudio por semana")
ax.set_ylabel("Calificacion final")
sns.despine()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
orden = df.groupby("carrera")["calificacion_final"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="carrera", y="calificacion_final", order=orden, color="#90a955", ax=ax)
sns.stripplot(data=df.sample(700, random_state=SEED), x="carrera", y="calificacion_final", order=orden, color="#132a13", alpha=0.18, size=3, ax=ax)
ax.set_title("Calificacion final por carrera")
ax.set_xlabel("")
ax.set_ylabel("Calificacion final")
ax.tick_params(axis="x", rotation=15)
sns.despine()
plt.show()

In [ ]:
num_cols = ["semestre", "horas_sueno", "horas_estudio", "asistencia_pct", "actividad_fisica_min", "cafe_tazas", "estres_escala", "uso_redes_horas", "promedio_previo", "trabaja", "calificacion_final"]
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, mask=mask, cmap="vlag", center=0, annot=True, fmt=".2f", linewidths=0.5, cbar_kws={"shrink": 0.75}, ax=ax)
ax.set_title("Mapa de correlaciones: senales para formular modelos")
plt.show()

## 5. Regresion lineal: modelo interpretable

Modelamos `calificacion_final` con predictores que tendrian sentido en una discusion academica: estudio, asistencia, sueno, estres, promedio previo y trabajo remunerado.

La interpretacion debe hacerse **manteniendo constantes** las demas variables del modelo.

In [ ]:
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error

formula_lm = "calificacion_final ~ horas_estudio + asistencia_pct + horas_sueno + I(horas_sueno**2) + estres_escala + promedio_previo + trabaja + C(carrera) + C(semestre)"
modelo_lm = smf.ols(formula_lm, data=df).fit()
print(modelo_lm.summary())

pred_lm = modelo_lm.predict(df)
try:
    rmse_lm = mean_squared_error(df["calificacion_final"], pred_lm, squared=False)
except TypeError:
    rmse_lm = mean_squared_error(df["calificacion_final"], pred_lm) ** 0.5
print(f"R2: {modelo_lm.rsquared:.3f}")
print(f"RMSE: {rmse_lm:.2f} puntos")

In [ ]:
coef_interes = modelo_lm.params[["horas_estudio", "asistencia_pct", "estres_escala", "promedio_previo", "trabaja"]]
interpretacion = pd.DataFrame({
    "coeficiente": coef_interes,
    "lectura_rapida": [
        "Cambio esperado en calificacion por una hora adicional de estudio semanal.",
        "Cambio esperado por un punto porcentual adicional de asistencia.",
        "Cambio esperado por un punto adicional en la escala de estres.",
        "Asociacion con historial academico previo.",
        "Cambio esperado por una hora adicional de trabajo remunerado semanal."
    ]
})
interpretacion.round(3)

In [ ]:
residuales = df["calificacion_final"] - pred_lm
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=pred_lm, y=df["calificacion_final"], alpha=0.25, s=24, color="#31572c", ax=axes[0])
lim = [df["calificacion_final"].min(), df["calificacion_final"].max()]
axes[0].plot(lim, lim, color="#bc4749", linewidth=2)
axes[0].set_title("Observado vs. predicho")
axes[0].set_xlabel("Prediccion del modelo")
axes[0].set_ylabel("Calificacion observada")
sns.scatterplot(x=pred_lm, y=residuales, alpha=0.25, s=24, color="#4f772d", ax=axes[1])
axes[1].axhline(0, color="#bc4749", linewidth=2)
axes[1].set_title("Residuales vs. prediccion")
axes[1].set_xlabel("Prediccion del modelo")
axes[1].set_ylabel("Residual")
sns.despine()
plt.tight_layout()
plt.show()

## 6. Clasificacion: probabilidad de aprobar

Ahora cambiamos la pregunta: no predecimos puntos de calificacion, sino la probabilidad de aprobar. La regresion logistica conecta con cursos de modelos lineales generalizados.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

features_num = ["horas_estudio", "asistencia_pct", "horas_sueno", "estres_escala", "promedio_previo", "trabaja", "uso_redes_horas"]
features_cat = ["carrera", "beca", "genero", "grupo"]
X = df[features_num + features_cat]
y = df["aprobacion"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
preprocess = ColumnTransformer([("num", StandardScaler(), features_num), ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat)])
logit = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000, random_state=SEED))])
logit.fit(X_train, y_train)
proba = logit.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print(f"Accuracy: {accuracy_score(y_test, pred):.3f}")
print(f"AUC ROC: {roc_auc_score(y_test, proba):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["No aprueba", "Aprueba"], cmap="Greens", ax=axes[0], colorbar=False)
axes[0].set_title("Matriz de confusion")
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1], color="#31572c", linewidth=3)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="#6c757d")
axes[1].set_title("Curva ROC")
plt.tight_layout()
plt.show()

In [ ]:
ejemplos = X_test.copy().assign(aprobacion_real=y_test.values, prob_aprobar=proba).sort_values("prob_aprobar", ascending=False)
ejemplos[["horas_estudio", "asistencia_pct", "horas_sueno", "estres_escala", "promedio_previo", "trabaja", "carrera", "aprobacion_real", "prob_aprobar"]].head(10)

## 7. Bootstrap: intervalo de confianza visual

El bootstrap permite aproximar la incertidumbre de un estimador re-muestreando la muestra observada. Aqui construiremos un intervalo de confianza para la media de `calificacion_final`.

In [ ]:
B = 1500
boot_means = np.empty(B)
values = df["calificacion_final"].to_numpy()
for b in range(B):
    boot_sample = rng.choice(values, size=len(values), replace=True)
    boot_means[b] = boot_sample.mean()
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
media_obs = values.mean()
print(f"Media observada: {media_obs:.2f}")
print(f"IC bootstrap 95%: [{ci_low:.2f}, {ci_high:.2f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.histplot(boot_means, bins=38, kde=True, color="#90a955", ax=ax)
ax.axvline(media_obs, color="#132a13", linewidth=3, label=f"Media observada = {media_obs:.2f}")
ax.axvspan(ci_low, ci_high, color="#31572c", alpha=0.16, label="IC bootstrap 95%")
ax.set_title("Distribucion bootstrap de la media de calificacion final")
ax.set_xlabel("Media bootstrap")
ax.set_ylabel("Frecuencia")
ax.legend()
sns.despine()
plt.show()

## 8. Seccion R companion

Para evitar depender de `rpy2` en el kernel Python, esta demo incluye un script companion completo:

`examples/academic/00_demo_estadistica_mrkov.R`

Puede abrirse en JupyterLab con kernel R o ejecutarse desde una terminal con:

```bash
Rscript examples/academic/00_demo_estadistica_mrkov.R
```

El script lee el mismo CSV, produce graficas con `ggplot2`, ajusta `lm()`, ajusta `glm(family = binomial)` e incluye bootstrap con `boot`.

## 9. Mini Spark local opcional

Esta celda usa Spark solo si `pyspark` esta disponible. Es deliberadamente local:

```python
SparkSession.builder.master("local[*]")
```

No lanza ejecutores distribuidos en Kubernetes. Es una demo ligera para mostrar interoperabilidad y el uso de `workers` en paralelo.

In [ ]:
try:
    from pyspark.sql import SparkSession
    import pyspark.sql.functions as F
    spark = (SparkSession.builder.master("local[*]").appName("mrkov-demo-estadistica-local").config("spark.ui.showConsoleProgress", "false").getOrCreate())
    sdf = spark.read.csv(str(DATA_PATH), header=True, inferSchema=True)
    resumen_spark = (sdf.groupBy("carrera", "semestre").agg(F.count("*").alias("n"), F.round(F.avg("calificacion_final"), 2).alias("media_calificacion"), F.round(F.avg("asistencia_pct"), 2).alias("media_asistencia")).orderBy("carrera", "semestre").toPandas())
    display(resumen_spark.head(12))
    spark.stop()
except Exception as exc:
    print("Spark local no disponible o no configurado en este entorno. La demo principal no depende de Spark.")
    print(type(exc).__name__, exc)

## 10. Cierre: que aprendimos

- Un entorno JupyterHub permite que toda la clase trabaje con los mismos datos y dependencias desde el navegador.
- Python y R pueden convivir en el flujo docente: exploracion, modelos, visualizacion e inferencia.
- Los modelos estadisticos no son cajas negras: sus coeficientes se conectan con preguntas academicas concretas.
- El bootstrap ayuda a explicar incertidumbre de forma computacional y visual.
- Spark puede mostrarse en modo local para una demo estable multiusuario; el procesamiento distribuido debe reservarse para practicas planeadas.

**Buenas practicas para clases y tesis**

- Guardar notebooks base en `Clases` y pedir a estudiantes copiarlos a su espacio personal antes de editarlos.
- Compartir datasets curados en `Repositorio/Datasets` o en una carpeta comun documentada.
- Mantener semillas, rutas relativas y versiones visibles dentro del notebook.
- Separar datos sensibles de datos de demostracion; para docencia publica, preferir datos sinteticos o anonimizados.